# 第13章

In [ ]:
!pip install dowhy==0.11
!pip install transformers==4.38.2
!pip install accelerate==0.28.0
!pip install pandas==2.0.3
!pip install numpy==1.25.2
!pip install pyro-ppl==1.9.0

## リスト13.1

In [ ]:
import numpy as np
import pandas as pd
import dowhy
from dowhy import CausalModel
from dowhy.datasets import linear_dataset

n_points = 1000
data = pd.DataFrame({
    "S": np.random.binomial(n=1, p=0.5, size=n_points),
    "LC": np.random.binomial(n=1, p=0.5, size=n_points),
    "Price": np.random.normal(loc=5, scale=1, size=n_points),
})

model=CausalModel(
        data = data,
        treatment='S',
        outcome='LC',
        common_causes=['G', 'A', 'E', 'O'],
        instruments=['Price']
)

identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

estimate = model.estimate_effect(identified_estimand,
                                 method_name="iv.instrumental_variable",
                                 method_params={'iv_instrument_name': 'Price'})

print(estimate)

## リスト13.2

In [ ]:
from transformers import GPT2Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokens = tokenizer.tokenize("Can LLMs reason counterfactually?")
print(tokens)

## リスト13.3

In [ ]:
input_ids = tokenizer.encode(
    "Can LLMs reason counterfactually?",
    return_tensors='pt'
)
print(input_ids)

## リスト13.4

In [ ]:
import torch
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained('gpt2-medium')
model.eval()

input_text = "Can LLMs reason counterfactually?<|endoftext|>"
input_ids = tokenizer.encode(input_text, return_tensors='pt')

with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
for idx, token in enumerate(input_ids[0]):
    token_log_prob = log_probs[0][idx][token].item()
    print(f"Token: {tokenizer.decode(token)}" +
           " | Log Probability: {token_log_prob}")

## リスト13.5

In [ ]:
prompt = "Counterfactual reasoning would enable AI to"
input_ids = tokenizer.encode(prompt, return_tensors='pt')

output = model.generate(
    input_ids,
    max_length=25,
    do_sample=True,
    pad_token_id=tokenizer.eos_token_id
)

generated_text = tokenizer.decode(
    output[0], skip_special_tokens=True)
print(generated_text)

## リスト13.6

In [ ]:
import pandas as pd
url = ("https://raw.githubusercontent.com/altdeep/"
       "causalAI/master/book/chapter%2013/"
       "king-prince-kingdom-updated.csv")
df = pd.read_csv(url)
print(df.shape[0])

print(df["King"][0] + "\n")
print(df["King"][1] + "\n")
print(df["King"][2])

print("----")
print(df["Prince"][0] + "\n")
print(df["Prince"][1] + "\n")
print(df["Prince"][2])

print("----")
print(df["Kingdom"][0] + "\n")
print(df["Kingdom"][1] + "\n")
print(df["Kingdom"][2])

## リスト13.7

In [ ]:
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    AutoTokenizer, DataCollatorForLanguageModeling,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    Trainer, TrainingArguments)
url = ("https://raw.githubusercontent.com/altdeep/"
       "causalAI/master/book/chapter%2013/"
       "king-prince-kingdom-updated.csv")
df = pd.read_csv(url)

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
tokenizer.pad_token = tokenizer.eos_token
def tokenize_phrases(phrases, max_length=40):
    return tokenizer(
        phrases,
        truncation=True,
        padding='max_length',
        max_length=max_length
    )

## リスト13.8

In [ ]:
class ModelDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings.input_ids)
def create_king_dataset(input_phrases):
    king_phrases = input_phrases.tolist()
    king_encodings = tokenize_phrases(king_phrases)
    king_dataset = ModelDataset(
        king_encodings, king_encodings['input_ids'])
    return king_dataset

## リスト13.9

In [ ]:
def create_seq2seq_datasets(input_phrases, target_phrases):
    input_phrases_list = input_phrases.tolist()
    target_phrases_list = target_phrases.tolist()
    spit = train_test_split(
        input_phrases_list,
        target_phrases_list,
        test_size=0.1
    )
    train_inputs, val_inputs, train_targets, val_targets = spit
    train_input_encodings = tokenize_phrases(train_inputs)
    val_input_encodings = tokenize_phrases(val_inputs)
    train_target_encodings = tokenize_phrases(train_targets)
    val_target_encodings = tokenize_phrases(val_targets)
    train_dataset = ModelDataset(
        train_input_encodings, train_target_encodings['input_ids']
    )
    val_dataset = ModelDataset(
        val_input_encodings, val_target_encodings['input_ids']
    )
    return train_dataset, val_dataset

## リスト13.10

In [ ]:
def train_king_model(output_dir, train_dataset,
                     model_name="gpt2-medium", epochs=4):
    king_model = AutoModelForCausalLM.from_pretrained(model_name)
    training_args_king = TrainingArguments(
      output_dir=output_dir,
      per_device_train_batch_size=32,
      overwrite_output_dir=True,
      num_train_epochs=epochs,
      save_total_limit=1,
      save_steps=len(train_dataset) // 16,
      max_grad_norm=1.0
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=False)
    trainer_king = Trainer(
        model=king_model,
        args=training_args_king,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )
    trainer_king.train()
    king_model.save_pretrained(output_dir)
    return king_model

## リスト13.11

In [ ]:
def train_seq2seq_model(output_dir, train_dataset, val_dataset,
                        model_name="facebook/bart-base",
                        epochs=4):
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=16,
        predict_with_generate=True,
        logging_dir=f"{output_dir}/logs",
        save_total_limit=1,
        save_steps=len(train_dataset) // 16,
        learning_rate=3e-5,
        num_train_epochs=epochs,
        warmup_steps=500,
        weight_decay=0.01,
    )
    trainer = Seq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )
    trainer.train()
    model.save_pretrained(output_dir)
    return model

## リスト13.12

In [ ]:
import os

king_model_path = os.path.join(os.getcwd(), 'king_model')
prince_model_path = os.path.join(os.getcwd(), 'prince_model')
kingdom_model_path = os.path.join(os.getcwd(), 'kingdom_model')
prince2king_model_path = os.path.join(
    os.getcwd(), 'prince2king_model')

king_dataset = create_king_dataset(df["King"])
king_model = train_king_model(king_model_path, king_dataset)

datasets = create_seq2seq_datasets(df["King"], df["Prince"])
train_dataset_prince, val_dataset_prince = datasets
prince_model = train_seq2seq_model(
    prince_model_path,
    train_dataset_prince,
    val_dataset_prince,
    epochs=6
)

king_and_prince = [f"{k} {p}" for k, p in zip(df["King"], df["Prince"])]
df["King and Prince"] = king_and_prince
train_dataset_kingdom, val_dataset_kingdom = create_seq2seq_datasets(
    df["King and Prince"], df["Kingdom"]
)
kingdom_model = train_seq2seq_model(
    kingdom_model_path,
    train_dataset_kingdom,
    val_dataset_kingdom,
   epochs=6
)

## リスト13.13

In [ ]:
p2k_data = create_seq2seq_datasets(
    df["Prince"], df["King"])
train_dataset_prince2king, val_dataset_prince2king = p2k_data
prince2king_model = train_seq2seq_model(
    prince2king_model_path,
    train_dataset_prince2king,
    val_dataset_prince2king,
    epochs=6
)

## リスト13.14

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import (
    AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    AutoTokenizer, GPT2LMHeadModel,
    PreTrainedModel, BartForConditionalGeneration)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

king_model = AutoModelForCausalLM.from_pretrained(
    "osazuwa/causalLLM-king").to(DEVICE)
prince_model = AutoModelForSeq2SeqLM.from_pretrained(
    "osazuwa/causalLLM-prince").to(DEVICE)
kingdom_model = AutoModelForSeq2SeqLM.from_pretrained(
    "osazuwa/causalLLM-kingdom").to(DEVICE)
prince2king_model = AutoModelForSeq2SeqLM.from_pretrained(
    "osazuwa/causalLLM-prince2king").to(DEVICE)

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
tokenizer.pad_token = tokenizer.eos_token

## リスト13.15

In [ ]:
def encode(text:str, device=DEVICE) -> torch.tensor:
    input_ids = tokenizer.encode(text, return_tensors="pt")
    input_ids = input_ids.to(device)
    return input_ids

def decode(text_ids: torch.tensor) -> str:
    output = tokenizer.decode(text_ids, skip_special_tokens=True)
    return output

EMPTY_TEXT = torch.tensor(tokenizer.encode("")).unsqueeze(0).to(DEVICE)

def generate_from_model(model: PreTrainedModel,
                        input_sequence: torch.tensor = EMPTY_TEXT,
                        max_length: int = 25,
                        temperature=1.0):
    output = model.generate(
        input_sequence,
        max_length=max_length,
        do_sample=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.pad_token_id,
        temperature=temperature,
        top_p=0.9,
    )
    return output

def convert_to_text(output):
    return decode(output[0]).strip().capitalize()

## リスト13.16

In [ ]:
def compute_log_probs(model, output_sequence):
    if isinstance(model, GPT2LMHeadModel):
        outputs = model(
            input_ids=output_sequence,
            labels=output_sequence
        )
        log_softmax = torch.nn.functional.log_softmax(
            outputs.logits, dim=-1)
        log_probs = log_softmax.gather(2, output_sequence.unsqueeze(-1))
        log_probs = log_probs.squeeze(-1).sum(dim=-1)
    elif isinstance(model, BartForConditionalGeneration):
        outputs = model(
            input_ids=output_sequence,
            labels=output_sequence)
        loss = outputs.loss
        log_probs = -loss * output_sequence.size(1)
    else:
        raise ValueError("Unsupported model type")
    return torch.tensor(log_probs.item())

## リスト13.17

In [ ]:
king_output = generate_from_model(king_model)
king_statement = convert_to_text(king_output)
print("Generated from king_model:", king_statement)
log_prob_king = compute_log_probs(king_model, king_output)
print("Log prob of generated king text:", log_prob_king)

prince_output = generate_from_model(prince_model, king_output)
prince_statement = convert_to_text(prince_output)
print("Generated from prince_model:", prince_statement)
log_prob_prince = compute_log_probs(prince_model, prince_output)
print("Log prob of generated prince text:", log_prob_prince)

king_prince_statement = king_statement + ". " + prince_statement
king_prince_output = encode(king_prince_statement)
kingdom_output = generate_from_model(kingdom_model, king_prince_output)
kingdom_statement = convert_to_text(kingdom_output)

print("Generated from kingdom model:", kingdom_statement)
log_prob_kingdom = compute_log_probs(kingdom_model, kingdom_output)
print("Log prob of generated kingdom text:", log_prob_kingdom)

king_output_infer = generate_from_model(prince2king_model, prince_output)
king_statement_infer = convert_to_text(king_output_infer)
print("Generated statement from prince2king:", king_statement_infer)
log_prob_prince2king = compute_log_probs(prince2king_model, prince_output)
print("Log prob of generated inference text:", log_prob_prince2king)

## リスト13.18

In [ ]:
import pyro
from pyro.distributions.torch_distribution \
import TorchDistributionMixin

class TransformerModelDistribution(TorchDistributionMixin):
    def __init__(self, model: PreTrainedModel,
                 input_encoding: torch.tensor = EMPTY_TEXT,
                ):
        super().__init__()
        self.model = model
        self.input_encoding = input_encoding

    def sample(self, sample_shape=torch.Size()):
        output = generate_from_model(
            self.model, self.input_encoding
        )
        return output

    def log_prob(self, value):
        return compute_log_probs(self.model, value)

## リスト13.19

In [ ]:
def causalLLM():
    king = pyro.sample(
        "King", TransformerModelDistribution(king_model)
    )
    prince = pyro.sample(
        "Prince", TransformerModelDistribution(
            prince_model, king)
    )
    king_and_prince = torch.cat([king, prince], dim=1)
    kingdom = pyro.sample(
        "Kingdom", TransformerModelDistribution(
            kingdom_model, king_and_prince)
    )
    king_text = convert_to_text(king)
    prince_text = convert_to_text(prince)
    kingdom_text = convert_to_text(kingdom)
    return king_text, prince_text, kingdom_text

for _ in range(2):
    king, prince, kingdom = causalLLM()
    vignette = " ".join([king, prince, kingdom])
    print(vignette)

## リスト13.20

In [ ]:
import pyro.poutine as poutine
from pyro.distributions import Categorical

PRINCE_STORY = (
    "His courageous Prince takes command, leading "
    "the kingdom's army to victory in battle after battle")
cond_model = pyro.condition(
    causalLLM, {"Prince": encode(PRINCE_STORY)})

def proposal_given_prince():
    prince = encode(PRINCE_STORY)
    king = pyro.sample(
        "King",
        TransformerModelDistribution(prince2king_model, prince)
    )
    king_and_prince = torch.cat([king, prince], dim=1)
    kingdom = pyro.sample(
        "Kingdom",
        TransformerModelDistribution(kingdom_model, king_and_prince)
    )
    vignette = (convert_to_text(king) +
        PRINCE_STORY +
        convert_to_text(kingdom))
    return vignette

## リスト13.21

In [ ]:
def process_sample(model, proposal):
    sample_trace = poutine.trace(proposal).get_trace()
    king_text = convert_to_text(sample_trace.nodes['King']['value'])
    kingdom_text = convert_to_text(
        sample_trace.nodes['Kingdom']['value'])
    proposal_log_prob = sample_trace.log_prob_sum()
    replay = poutine.replay(model, trace=sample_trace)
    model_trace = poutine.trace(replay).get_trace()
    model_log_prob = model_trace.log_prob_sum()
    log_importance_weight = model_log_prob - proposal_log_prob
    sample = (king_text, kingdom_text, log_importance_weight)
    return sample

## リスト13.22

In [ ]:
def do_importance_resampling(model, proposal, num_samples):
    original_samples = []
    for _ in range(num_samples):
        sample = process_sample(model, proposal)
        original_samples.append(sample)
    unique_samples = list(set(original_samples))
    log_importance_weights = torch.tensor(
        [sample[2] for sample in original_samples])
    resampling_dist = Categorical(logits=log_importance_weights)
    resampled_indices = resampling_dist.sample_n(num_samples)
    samples = pd.DataFrame(
        [unique_samples[i] for i in resampled_indices],
        columns=["King", "Kingdom", "log_importance_weight"]
    )
    samples["Prince"] = PRINCE_STORY
    samples["Distribution"] = "observational"
    return samples[['King', 'Prince', "Kingdom", "Distribution"]]

num_samples = 1000
posterior_samples = do_importance_resampling(
    cond_model, proposal_given_prince, num_samples)

## リスト13.23

In [ ]:
intervention_model = pyro.do(
    causalLLM, {"Prince": encode(PRINCE_STORY)})
intervention_samples = pd.DataFrame(
    [intervention_model() for _ in range(num_samples)],
    columns=["King", "Prince", "Kingdom"]
)
intervention_samples["Distribution"] = "interventional"
all_samples = pd.concat(
    [posterior_samples, intervention_samples],
    ignore_index=True
)

## リスト13.24

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

kingdom_samples_url = (
    "https://raw.githubusercontent.com/altdeep/causalAI/"
    "master/book/chapter%2013/kingdom_samples.csv")
all_samples = pd.read_csv(kingdom_samples_url)

observational_texts = all_samples[
    all_samples["Distribution"] == "observational"]["Kingdom"]
interventional_texts = all_samples[all_samples[
    "Distribution"] == "interventional"]["Kingdom"]

vectorizer = TfidfVectorizer(stop_words='english')
X_obs = vectorizer.fit_transform(observational_texts)
X_int = vectorizer.transform(interventional_texts)

k = 10
feature_names = vectorizer.get_feature_names_out()
obs_indices = X_obs.sum(axis=0).argsort()[0, -k:][::-1]
int_indices = X_int.sum(axis=0).argsort()[0, -k:][::-1]
combined_indices = np.concatenate((obs_indices, int_indices))
combined_indices = np.unique(combined_indices)

## リスト13.25

In [ ]:
import matplotlib.pyplot as plt

labels = [feature_names[i] for i in combined_indices]
labels, indices = np.unique(labels, return_index=True)
obs_values = np.array(X_obs.sum(axis=0))[0, combined_indices]
int_values = np.array(X_int.sum(axis=0))[0, combined_indices]
obs_values = [obs_values[0][i] for i in indices]
int_values = [int_values[0][i] for i in indices]
combined = list(zip(labels, obs_values, int_values))
sorted_combined = sorted(combined, key=lambda x: (-x[1], x[2]))
labels, obs_values, int_values = zip(*sorted_combined)

width = 0.35
x = np.arange(len(labels))
fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, obs_values, width,
                label='Observational', alpha=0.7)
rects2 = ax.bar(x + width/2, int_values, width,
                label='Interventional', alpha=0.7)
ax.set_xlabel('Words')
ax.set_ylabel('TF-IDF Values')
ax.set_title('Top Words in Generated Kingdom Vignettes by TF-IDF Value')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
fig.tight_layout()
plt.xticks(rotation=45)
plt.show()